# Binary Neural Network From Scratch: Clean vs Unclean

This notebook is the binary version of the neural network notebook.

The model predicts only one thing:

- `clean` means the comment has none of the six warning labels.
- `unclean` means the comment has at least one of the six warning labels.

The model output is one number. After sigmoid, that number becomes the probability that the comment is `unclean`.


## Install Modules


In [2]:
# Install the modules used in this notebook
# If they are already installed, this will say Requirement already satisfied
%pip install pandas numpy scikit-learn torch


  Using cached torch-2.12.0-cp314-cp314-win_amd64.whl.metadata (31 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.3 MB 3.0 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.3 MB 3.0 MB/s eta 0:00:03
   ----------- ---------------------------- 2.4/8.3 MB 3.1 MB/s eta 0:00:02
   --------------- -----------------------

## Imports


In [3]:
import pandas as pd
import numpy as np
import re
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


## Load Files And Make Clean/Unclean Labels


In [4]:
# Load files from the current folder
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
test_labels = pd.read_csv("test_labels.csv")

# Remove rows where comment_text is missing
train_df = train_df.dropna(subset=["comment_text"])
test_df = test_df.dropna(subset=["comment_text"])

# Make sure comment_text is stored as text
train_df["comment_text"] = train_df["comment_text"].astype(str)
test_df["comment_text"] = test_df["comment_text"].astype(str)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Test labels shape:", test_labels.shape)

# These are the original six label columns from the dataset
label_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Count how many of the original labels each comment has
train_df["label_count"] = train_df[label_cols].sum(axis=1)

# unclean = 1 means the comment has at least one original label
train_df["unclean"] = (train_df["label_count"] > 0).astype(int)

# clean = 1 means the comment has no original labels
train_df["clean"] = 1 - train_df["unclean"]

# Show how many comments are clean and unclean
print("Clean comments:", train_df["clean"].sum())
print("Unclean comments:", train_df["unclean"].sum())
print("Comments with more than one original label:", (train_df["label_count"] > 1).sum())


Train shape: (159571, 8)
Test shape: (153164, 2)
Test labels shape: (153164, 7)
Clean comments: 143346
Unclean comments: 16225
Comments with more than one original label: 9865


## Clean Text


In [5]:
# This function makes text easier for the model to read
def clean_text(text):
    # Convert text to lowercase so Hello and hello are treated the same
    text = text.lower()

    # Keep only letters, numbers, and spaces
    text = re.sub(r"[^a-z0-9 ]", " ", text)

    # Replace repeated spaces with one space
    text = re.sub(r"\s+", " ", text).strip()

    # Return the cleaned comment
    return text


# Clean training comments
train_df["clean_text"] = train_df["comment_text"].apply(clean_text)

# Clean test comments
test_df["clean_text"] = test_df["comment_text"].apply(clean_text)


## Split Training Data


In [6]:
# Split train.csv into training data and validation data
# stratify keeps a similar clean/unclean ratio in both parts
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["unclean"]
)

# Reset indexes after splitting
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

# Check split sizes
print("Training data:", train_data.shape)
print("Validation data:", val_data.shape)

# Check clean/unclean balance in both splits
print("Training clean percent:", train_data["clean"].mean() * 100)
print("Training unclean percent:", train_data["unclean"].mean() * 100)
print("Validation clean percent:", val_data["clean"].mean() * 100)
print("Validation unclean percent:", val_data["unclean"].mean() * 100)


Training data: (127656, 12)
Validation data: (31915, 12)
Training clean percent: 89.832048630695
Training unclean percent: 10.167951369305008
Validation clean percent: 89.83236722544258
Validation unclean percent: 10.167632774557418


## Tokenizer


In [7]:
# This function splits a comment into words
def tokenize(text):
    # Split text by spaces
    return text.split()


## Build Vocabulary


In [8]:
# Maximum number of words we keep
MAX_VOCAB_SIZE = 50000

# Counter stores how many times each word appears
counter = Counter()

# Count words in the training comments only
for text in train_data["clean_text"]:
    # Split comment into words
    words = tokenize(text)

    # Add word counts
    counter.update(words)

# Create vocabulary with two special tokens
vocab = {
    "<PAD>": 0,   # Used to fill short comments
    "<UNK>": 1    # Used for words not in the vocabulary
}

# Add the most common words to the vocabulary
for word, count in counter.most_common(MAX_VOCAB_SIZE - 2):
    # Give each word a unique number
    vocab[word] = len(vocab)

# Print vocabulary size
print("Vocabulary size:", len(vocab))


Vocabulary size: 50000


## Convert Text To Numbers


In [9]:
# Every comment will become exactly 200 word IDs long
MAX_LEN = 200

# This function converts one comment into numbers
def encode_text(text):
    # Split comment into words
    words = tokenize(text)

    # Convert each word to its vocabulary number
    ids = [vocab.get(word, vocab["<UNK>"]) for word in words]

    # If comment is shorter than 200 words, add padding
    if len(ids) < MAX_LEN:
        ids = ids + [vocab["<PAD>"]] * (MAX_LEN - len(ids))

    # If comment is longer than 200 words, cut it to 200 words
    else:
        ids = ids[:MAX_LEN]

    # Return list of word numbers
    return ids


## Create Dataset Class


In [10]:
# This class prepares comments and labels for PyTorch
class CleanUncleanDataset(Dataset):

    # This runs when we create the dataset
    def __init__(self, dataframe):
        # Store cleaned comments
        self.texts = dataframe["clean_text"].values

        # Store the binary target
        # 0 means clean, 1 means unclean
        self.labels = dataframe["unclean"].values.astype(np.float32)

    # This returns the number of rows
    def __len__(self):
        return len(self.texts)

    # This returns one comment and one label
    def __getitem__(self, idx):
        # Convert one comment into word IDs
        x = encode_text(self.texts[idx])

        # Convert word IDs into a PyTorch tensor
        x = torch.tensor(x, dtype=torch.long)

        # Put the label inside a list so it matches the model output shape
        y = torch.tensor([self.labels[idx]], dtype=torch.float)

        # Return comment and label
        return x, y


## Create Data Loaders


In [11]:
# Batch size means how many comments the model sees at once
BATCH_SIZE = 128

# Create training dataset
train_dataset = CleanUncleanDataset(train_data)

# Create validation dataset
val_dataset = CleanUncleanDataset(val_data)

# Create training loader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Create validation loader
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


## Simple Binary Neural Network


In [12]:
class SimpleBinaryTextNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        # Converts word IDs into word vectors the model can learn from
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # First hidden layer
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)

        # Dropout helps the model avoid memorizing the training data too much
        self.dropout = nn.Dropout(0.3)

        # Final output layer
        # It gives one raw number for clean vs unclean
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Convert word IDs to embeddings
        embedded = self.embedding(x)

        # Find which positions are real words and which are padding
        mask = (x != 0).unsqueeze(-1)

        # Set padding embeddings to zero so they do not affect the average
        embedded = embedded * mask

        # Add the real word embeddings together
        summed = embedded.sum(dim=1)

        # Count how many real words each comment has
        counts = mask.sum(dim=1).clamp(min=1)

        # Average the word embeddings into one comment vector
        pooled = summed / counts

        # Hidden layer
        hidden = F.relu(self.fc1(pooled))

        # Apply dropout
        hidden = self.dropout(hidden)

        # Final raw output
        logits = self.fc2(hidden)

        return logits


## Setup Model


In [13]:
# Use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create model
model = SimpleBinaryTextNN(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_dim=128
).to(device)

# Count unclean and clean examples in the training split
unclean_count = train_data["unclean"].sum()
clean_count = len(train_data) - unclean_count

# The dataset has many more clean comments than unclean comments
# pos_weight gives the unclean class extra importance during training
pos_weight_value = np.sqrt(clean_count / unclean_count)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float).to(device)

# Loss function for one yes/no output
# The model gives raw logits, and this loss applies sigmoid internally
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer updates model weights
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Print setup information
print("Using device:", device)
print("Clean count:", int(clean_count))
print("Unclean count:", int(unclean_count))
print("Unclean pos_weight:", round(float(pos_weight_value), 2))


Using device: cpu
Clean count: 114676
Unclean count: 12980
Unclean pos_weight: 2.97


## Training Function


In [14]:
# This function trains the model for one epoch
def train_one_epoch():
    # Put model in training mode
    model.train()

    # Store total loss
    total_loss = 0

    # Loop through batches
    for x, y in train_loader:
        # Move data to GPU/CPU
        x = x.to(device)
        y = y.to(device)

        # Clear old weight-change calculations
        optimizer.zero_grad()

        # Get model outputs
        logits = model(x)

        # Calculate loss
        loss = criterion(logits, y)

        # Work out how each weight should change
        loss.backward()

        # Apply those changes to the model weights
        optimizer.step()

        # Add batch loss
        total_loss += loss.item()

    # Return average loss
    return total_loss / len(train_loader)


## Evaluation Function


In [15]:
# This function collects model probabilities and real labels
def get_probs_and_labels(loader):
    # Put model in evaluation mode
    model.eval()

    # Store predicted probabilities
    all_probs = []

    # Store true labels
    all_labels = []

    # We are only checking the model, so no weight changes are needed
    with torch.no_grad():
        # Loop through batches
        for x, y in loader:
            # Move data to GPU/CPU
            x = x.to(device)
            y = y.to(device)

            # Get raw model outputs
            logits = model(x)

            # Convert raw outputs into unclean probabilities
            probs = torch.sigmoid(logits)

            # Store this batch
            all_probs.append(probs.cpu().numpy().reshape(-1))
            all_labels.append(y.cpu().numpy().reshape(-1))

    # Combine all batches into one array
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels).astype(int)

    return all_probs, all_labels


# This function checks model performance
def evaluate(loader, threshold=0.5):
    # Get probabilities and real labels
    probs, labels = get_probs_and_labels(loader)

    # Convert probabilities into clean/unclean predictions
    preds = (probs >= threshold).astype(int)

    # AUC checks whether unclean comments usually get higher scores
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = np.nan

    # Confusion matrix uses this order: clean first, unclean second
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    clean_correct = cm[0, 0]
    clean_called_unclean = cm[0, 1]
    unclean_called_clean = cm[1, 0]
    unclean_correct = cm[1, 1]

    # Return the useful numbers in one dictionary
    return {
        "threshold": threshold,
        "auc": auc,
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "clean_correct": clean_correct,
        "clean_called_unclean": clean_called_unclean,
        "unclean_called_clean": unclean_called_clean,
        "unclean_correct": unclean_correct
    }


## Train Model


In [16]:
# Number of times model sees the full training data
EPOCHS = 10

# Best validation AUC starts very low
best_val_auc = -1

# File where the best binary model will be saved
best_model_path = "best_binary_clean_unclean_nn.pt"

# Loop through epochs
for epoch in range(EPOCHS):
    # Train model for one epoch
    train_loss = train_one_epoch()

    # Check validation performance using the default 0.5 threshold
    val_results = evaluate(val_loader, threshold=0.5)

    # Save model if validation AUC improves
    if val_results["auc"] > best_val_auc:
        best_val_auc = val_results["auc"]
        torch.save(model.state_dict(), best_model_path)

    # Print results
    print("Epoch:", epoch + 1)
    print("Train loss:", round(train_loss, 4))
    print("Validation AUC:", round(val_results["auc"], 4))
    print("Validation Accuracy:", round(val_results["accuracy"], 4))
    print("Validation Precision:", round(val_results["precision"], 4))
    print("Validation Recall:", round(val_results["recall"], 4))
    print("Validation F1:", round(val_results["f1"], 4))
    print("-------------------------")

# Print best validation AUC
print("Best Validation AUC:", best_val_auc)


Epoch: 1
Train loss: 0.4029
Validation AUC: 0.9399
Validation Accuracy: 0.9271
Validation Precision: 0.6228
Validation Recall: 0.7165
Validation F1: 0.6664
-------------------------
Epoch: 2
Train loss: 0.2488
Validation AUC: 0.9577
Validation Accuracy: 0.9405
Validation Precision: 0.6862
Validation Recall: 0.7643
Validation F1: 0.7231
-------------------------
Epoch: 3
Train loss: 0.1845
Validation AUC: 0.9613
Validation Accuracy: 0.9437
Validation Precision: 0.6971
Validation Recall: 0.7892
Validation F1: 0.7403
-------------------------
Epoch: 4
Train loss: 0.1447
Validation AUC: 0.9621
Validation Accuracy: 0.9459
Validation Precision: 0.7089
Validation Recall: 0.7932
Validation F1: 0.7487
-------------------------
Epoch: 5
Train loss: 0.1175
Validation AUC: 0.9606
Validation Accuracy: 0.9503
Validation Precision: 0.7484
Validation Recall: 0.7707
Validation F1: 0.7594
-------------------------
Epoch: 6
Train loss: 0.097
Validation AUC: 0.9578
Validation Accuracy: 0.9502
Validation P

## Find Best Threshold


In [17]:
# This function finds the threshold that gives the best F1 score
def find_best_threshold(loader):
    # Get validation probabilities and labels
    probs, labels = get_probs_and_labels(loader)

    # Start with the normal 0.5 threshold
    best_threshold = 0.5
    best_f1 = 0

    # Try thresholds from 0.05 to 0.95
    for threshold in np.arange(0.05, 0.96, 0.05):
        # Make clean/unclean predictions at this threshold
        preds = (probs >= threshold).astype(int)

        # Calculate F1 score
        f1 = f1_score(labels, preds, zero_division=0)

        # Keep the threshold if it improves F1
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    # Return the best threshold
    return best_threshold


# Load the best saved model before choosing the threshold
model.load_state_dict(torch.load(best_model_path, map_location=device))

# Find best threshold on validation data
best_threshold = find_best_threshold(val_loader)

# Evaluate validation data with the chosen threshold
val_results = evaluate(val_loader, threshold=best_threshold)

print("Best threshold:", round(best_threshold, 2))
print("Validation results with best threshold:")
pd.DataFrame([val_results])


Best threshold: 0.7
Validation results with best threshold:


,threshold,auc,accuracy,precision,recall,f1,clean_correct,clean_called_unclean,unclean_called_clean,unclean_correct
0,0.7,0.962066,0.954285,0.806452,0.724191,0.763111,28106,564,895,2350


## Prepare Real Test Data


In [18]:
# test_labels.csv has some rows with -1
# -1 means those rows should not be used for scoring
valid_test_labels = test_labels[
    (test_labels[label_cols] != -1).all(axis=1)
].copy()

# Build the same clean/unclean target for the real labeled test rows
valid_test_labels["label_count"] = valid_test_labels[label_cols].sum(axis=1)
valid_test_labels["unclean"] = (valid_test_labels["label_count"] > 0).astype(int)
valid_test_labels["clean"] = 1 - valid_test_labels["unclean"]

# Merge test comments with their clean/unclean labels
test_labeled_df = test_df.merge(
    valid_test_labels[["id", "clean", "unclean"]],
    on="id"
)

# Check test data size and balance
print("Real labeled test data:", test_labeled_df.shape)
print("Clean test comments:", test_labeled_df["clean"].sum())
print("Unclean test comments:", test_labeled_df["unclean"].sum())


Real labeled test data: (63978, 5)
Clean test comments: 57735
Unclean test comments: 6243


## Test Model


In [19]:
# Create test dataset with real labels
test_dataset = CleanUncleanDataset(test_labeled_df)

# Create test loader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Load the best saved model before testing
model.load_state_dict(torch.load(best_model_path, map_location=device))

# Evaluate on test data using the best validation threshold
test_results = evaluate(test_loader, threshold=best_threshold)

# Print final test results
print("Test AUC:", round(test_results["auc"], 4))
print("Test Accuracy:", round(test_results["accuracy"], 4))
print("Test Precision:", round(test_results["precision"], 4))
print("Test Recall:", round(test_results["recall"], 4))
print("Test F1:", round(test_results["f1"], 4))

# Show all test results in a table
pd.DataFrame([test_results])


Test AUC: 0.9482
Test Accuracy: 0.9075
Test Precision: 0.5165
Test Recall: 0.8158
Test F1: 0.6326


,threshold,auc,accuracy,precision,recall,f1,clean_correct,clean_called_unclean,unclean_called_clean,unclean_correct
0,0.7,0.948191,0.907515,0.516531,0.815794,0.632553,52968,4767,1150,5093


## Try One Comment


In [20]:
# This function predicts clean or unclean for one new comment
def predict_clean_unclean(comment, threshold=best_threshold):
    # Clean the new comment the same way as training comments
    cleaned = clean_text(comment)

    # Convert the comment into word IDs
    encoded = encode_text(cleaned)

    # Add a batch dimension because the model expects a batch
    x = torch.tensor([encoded], dtype=torch.long).to(device)

    # Make prediction
    model.eval()
    with torch.no_grad():
        logits = model(x)
        prob_unclean = torch.sigmoid(logits).item()

    # Pick the final label
    predicted_label = "unclean" if prob_unclean >= threshold else "clean"

    # Return a small readable result
    return {
        "comment": comment,
        "prob_unclean": prob_unclean,
        "predicted_label": predicted_label
    }


# Example clean comment
predict_clean_unclean("Thank you for improving the article.")


{'comment': 'Thank you for improving the article.',
 'prob_unclean': 0.0005843217950314283,
 'predicted_label': 'clean'}